Answer the problem in the context of topics covered in SDS. You are **not allowed** to seek or receive help from anyone to solve the problem. To keep the problem at reasonable difficulty, you are also **not allowed** to use LLMs such as ChatGPT, Bard/Gemini, Claude and Llama. You can **only use built-in libraries** of Apache Spark. For example, you cannot use Spark packages.

# Problem 3 [25 pts]

In this problem, you will be working with a network of power connections defined by `pc_nodes.csv` and `pc_edges.csv`.

### Revised setup cell for the attached real data files

The attached files are **headerless**:

- `pc_nodes (1).csv`: column 1 = node ID, column 2 = topic/type. Some topic values are blank.
- `pc_edges (1).csv`: column 1 = source node, column 2 = destination node.

This notebook therefore reads them with `header=False`. It also uses Spark RDD iterations for PageRank and HITS to avoid the Spark DataFrame query-plan memory issues encountered earlier.

In [ ]:
# ============================================================
# Spark setup and real CSV loading
# ============================================================
# Use the attached real files:
# - pc_nodes (1).csv
# - pc_edges (1).csv
# These files are headerless, so we read with header=False.

from pathlib import Path
import math
from pyspark.sql import SparkSession, functions as F, types as T

spark = (
    SparkSession.builder
    .appName("SDS_Problem3_Real_Data")
    .master("local[*]")
    .config("spark.driver.memory", "2g")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.sql.adaptive.enabled", "false")
    .config("spark.ui.enabled", "false")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")
sc = spark.sparkContext

# Try the exact uploaded filenames first, then fallback names.
NODE_PATH_CANDIDATES = [
    "/mnt/data/pc_nodes (1).csv",
    "pc_nodes (1).csv",
    "/mnt/data/pc_nodes.csv",
    "pc_nodes.csv",
]

EDGE_PATH_CANDIDATES = [
    "/mnt/data/pc_edges (1).csv",
    "pc_edges (1).csv",
    "/mnt/data/pc_edges.csv",
    "pc_edges.csv",
]

def choose_existing_path(candidates):
    for p in candidates:
        if Path(p).exists():
            return p
    # Return first candidate anyway so Spark shows a meaningful error if files are elsewhere.
    return candidates[0]

NODES_PATH = choose_existing_path(NODE_PATH_CANDIDATES)
EDGES_PATH = choose_existing_path(EDGE_PATH_CANDIDATES)

print("Using nodes file:", NODES_PATH)
print("Using edges file:", EDGES_PATH)

nodes = (
    spark.read
    .option("header", "false")
    .option("inferSchema", "true")
    .csv(NODES_PATH)
    .select(
        F.col("_c0").cast("string").alias("id"),
        F.col("_c1").cast("string").alias("topic")
    )
    .dropna(subset=["id"])
    .withColumn(
        "topic",
        F.when(F.col("topic").isNull() | (F.trim(F.col("topic")) == ""), F.lit("unknown"))
         .otherwise(F.col("topic"))
    )
    .dropDuplicates(["id"])
)

edges = (
    spark.read
    .option("header", "false")
    .option("inferSchema", "true")
    .csv(EDGES_PATH)
    .select(
        F.col("_c0").cast("string").alias("src"),
        F.col("_c1").cast("string").alias("dst")
    )
    .dropna(subset=["src", "dst"])
    .dropDuplicates(["src", "dst"])
)

# Keep only edges whose endpoints exist in the node table.
valid_src = nodes.select(F.col("id").alias("src"))
valid_dst = nodes.select(F.col("id").alias("dst"))

edges = (
    edges
    .join(valid_src, on="src", how="inner")
    .join(valid_dst, on="dst", how="inner")
    .dropDuplicates(["src", "dst"])
)

nodes.cache()
edges.cache()

print("Number of nodes:", nodes.count())
print("Number of edges:", edges.count())
print("Topic counts:")
nodes.groupBy("topic").count().orderBy(F.desc("count")).show(20, truncate=False)

print("Sample nodes:")
nodes.show(10, truncate=False)

print("Sample edges:")
edges.show(10, truncate=False)

## Problem 3a [10 pts]

Using Apache Spark, compute the PageRank of all nodes relative to `residential` nodes. Display the ID, topic and PageRank of the 25 nodes with the largest PageRank.

### Revised solution for Problem 3a

The phrase **“PageRank relative to residential nodes”** is implemented as **personalized PageRank**. The teleport/restart probability is placed only on nodes whose topic is `residential`.

This means the score answers:

> Which nodes are most important when random walks repeatedly restart from residential nodes?

Because the real graph has 10,000 nodes, the implementation below uses Spark **RDDs** for the iterative part. This avoids the long DataFrame lineage and Java heap errors that can happen in iterative graph algorithms.

In [ ]:
# ============================================================
# Problem 3a: Personalized PageRank relative to residential nodes
# ============================================================

DAMPING = 0.85
PAGERANK_ITERATIONS = 10

all_node_ids = [row["id"] for row in nodes.select("id").collect()]
N = len(all_node_ids)

residential_ids = [
    row["id"] for row in nodes
    .filter(F.lower(F.col("topic")) == "residential")
    .select("id")
    .collect()
]

R = len(residential_ids)

if N == 0:
    raise ValueError("No nodes found.")

if R == 0:
    raise ValueError("No residential nodes found. Check the topic column in pc_nodes (1).csv.")

print("Total nodes:", N)
print("Residential nodes:", R)

residential_set_bc = sc.broadcast(set(residential_ids))
N_bc = sc.broadcast(N)
R_bc = sc.broadcast(R)

# adjacency: (src, [dst1, dst2, ...])
adjacency = (
    edges.rdd
    .map(lambda row: (row["src"], row["dst"]))
    .distinct()
    .groupByKey()
    .mapValues(lambda xs: list(xs))
    .cache()
)

all_nodes_rdd = sc.parallelize(all_node_ids).map(lambda x: (x, None)).cache()

# Start from the personalized teleport distribution.
def teleport_prob(node_id):
    return 1.0 / R_bc.value if node_id in residential_set_bc.value else 0.0

ranks = sc.parallelize([(node_id, teleport_prob(node_id)) for node_id in all_node_ids]).cache()

for i in range(PAGERANK_ITERATIONS):
    # Nodes with adjacency lists emit rank contribution to outgoing neighbors.
    joined = ranks.leftOuterJoin(adjacency)

    def emit_contribs(item):
        node_id, (rank, nbrs) = item
        if nbrs is None or len(nbrs) == 0:
            return []
        share = rank / len(nbrs)
        return [(dst, share) for dst in nbrs]

    contribs = joined.flatMap(emit_contribs).reduceByKey(lambda a, b: a + b)

    # Dangling mass: rank from nodes with no outgoing edges.
    dangling_mass = joined.filter(lambda x: x[1][1] is None or len(x[1][1]) == 0).map(lambda x: x[1][0]).sum()

    # Redistribute dangling mass using the personalized teleport distribution.
    new_ranks = (
        all_nodes_rdd
        .leftOuterJoin(contribs)
        .map(lambda x: (
            x[0],
            (1.0 - DAMPING) * teleport_prob(x[0])
            + DAMPING * ((x[1][1] or 0.0) + dangling_mass * teleport_prob(x[0]))
        ))
        .cache()
    )

    ranks.unpersist()
    ranks = new_ranks
    print(f"Finished personalized PageRank iteration {i + 1}/{PAGERANK_ITERATIONS}")

pagerank_schema = T.StructType([
    T.StructField("id", T.StringType(), False),
    T.StructField("pagerank", T.DoubleType(), False),
])

pagerank_df = spark.createDataFrame(ranks, schema=pagerank_schema)

pagerank_top25 = (
    pagerank_df
    .join(nodes, on="id", how="left")
    .select("id", "topic", "pagerank")
    .orderBy(F.col("pagerank").desc())
    .limit(25)
)

pagerank_top25.show(25, truncate=False)

## Problem 3b [10 pts]

Using Apache Spark, compute the hubbiness and authority of all nodes. Display the ID, topic, hubbiness and authority of the 25 nodes with the largest hubbiness as well as the 25 nodes with the largest authority.

### Revised solution for Problem 3b

HITS computes two scores:

- **Hubbiness**: high if the node points to many good authorities.
- **Authority**: high if the node is pointed to by many good hubs.

The implementation below also uses Spark RDDs to avoid iterative DataFrame memory issues.

In [ ]:
# ============================================================
# Problem 3b: HITS hubbiness and authority using Spark RDDs
# ============================================================

HITS_ITERATIONS = 10

edge_pairs = edges.rdd.map(lambda row: (row["src"], row["dst"])).distinct().cache()
all_nodes_rdd = sc.parallelize(all_node_ids).cache()

# Initialize hub and authority scores to 1.0.
hubs = all_nodes_rdd.map(lambda node_id: (node_id, 1.0)).cache()
authorities = all_nodes_rdd.map(lambda node_id: (node_id, 1.0)).cache()

for i in range(HITS_ITERATIONS):
    # Authority update:
    # authority(dst) = sum hub(src) over incoming edges src -> dst
    authority_raw = (
        edge_pairs
        .join(hubs)  # (src, (dst, hub_src))
        .map(lambda x: (x[1][0], x[1][1]))
        .reduceByKey(lambda a, b: a + b)
    )

    auth_norm = math.sqrt(authority_raw.map(lambda x: x[1] * x[1]).sum()) or 1.0
    new_authorities = (
        all_nodes_rdd.map(lambda node_id: (node_id, 0.0))
        .leftOuterJoin(authority_raw)
        .map(lambda x: (x[0], (x[1][1] or 0.0) / auth_norm))
        .cache()
    )

    # Hub update:
    # hub(src) = sum authority(dst) over outgoing edges src -> dst
    edges_by_dst = edge_pairs.map(lambda x: (x[1], x[0]))  # (dst, src)
    hub_raw = (
        edges_by_dst
        .join(new_authorities)  # (dst, (src, authority_dst))
        .map(lambda x: (x[1][0], x[1][1]))
        .reduceByKey(lambda a, b: a + b)
    )

    hub_norm = math.sqrt(hub_raw.map(lambda x: x[1] * x[1]).sum()) or 1.0
    new_hubs = (
        all_nodes_rdd.map(lambda node_id: (node_id, 0.0))
        .leftOuterJoin(hub_raw)
        .map(lambda x: (x[0], (x[1][1] or 0.0) / hub_norm))
        .cache()
    )

    hubs.unpersist()
    authorities.unpersist()
    hubs = new_hubs
    authorities = new_authorities

    print(f"Finished HITS iteration {i + 1}/{HITS_ITERATIONS}")

hits_rdd = hubs.join(authorities).map(lambda x: (x[0], x[1][0], x[1][1]))

hits_schema = T.StructType([
    T.StructField("id", T.StringType(), False),
    T.StructField("hubbiness", T.DoubleType(), False),
    T.StructField("authority", T.DoubleType(), False),
])

hits_df = spark.createDataFrame(hits_rdd, schema=hits_schema)

hits_with_topics = (
    hits_df
    .join(nodes, on="id", how="left")
    .select("id", "topic", "hubbiness", "authority")
)

hits_top_hubbiness = hits_with_topics.orderBy(F.col("hubbiness").desc()).limit(25)
hits_top_authority = hits_with_topics.orderBy(F.col("authority").desc()).limit(25)

print("Top 25 by hubbiness:")
hits_top_hubbiness.show(25, truncate=False)

print("Top 25 by authority:")
hits_top_authority.show(25, truncate=False)

## Problem 3c [5 pts]

Based on the results of Problems 3a and 3b, give three pieces of advice you can give to the CEO of the power transmission company who owns the power connections. Back each advice with the appropriate previous result.

### Revised solution for Problem 3c

The advice below is backed by the three result tables computed above:

1. `pagerank_top25`
2. `hits_top_hubbiness`
3. `hits_top_authority`

In [ ]:
# ============================================================
# Problem 3c: Evidence-backed advice to the CEO
# ============================================================

print("Evidence for advice 1: highest personalized PageRank relative to residential nodes")
pagerank_top25.show(25, truncate=False)

print("Evidence for advice 2: highest hubbiness nodes")
hits_top_hubbiness.show(25, truncate=False)

print("Evidence for advice 3: highest authority nodes")
hits_top_authority.show(25, truncate=False)

print("""
CEO advice:

1. Prioritize reliability upgrades and preventive maintenance for the top personalized-PageRank nodes.
   These are the nodes most important from the perspective of residential connectivity. If they fail,
   they are more likely to affect residential service paths.

2. Monitor and protect the top hubbiness nodes as operational connector points.
   High-hubbiness nodes point to many strong authority nodes, so they are important for distributing
   flow or connectivity across the network.

3. Add redundancy or contingency planning around the top authority nodes.
   High-authority nodes are pointed to by strong hubs, so they may represent concentration points.
   If a high-authority node fails, it can disrupt many important upstream connector paths.
""")